# Lab: Query Optimization - Solutions

**Objective**: Master Spark query optimization techniques to write efficient data processing pipelines.

**Learning Outcomes**:
- Understand how Catalyst optimizer consolidates filters
- Recognize when caching helps vs hurts performance
- Identify predicate pushdown in explain plans
- Compare columnar (Parquet) vs row-based (CSV) formats
- Apply optimization principles to avoid common anti-patterns

**Estimated Time**: 45 minutes

---

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import time

spark = SparkSession.builder \
    .appName("Lab-Query-Optimization") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.logLevel", "ERROR") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(f"🚀 Spark {spark.version} - Query Optimization Lab")
ui_url = spark.sparkContext.uiWebUrl
print(f"Spark UI: {ui_url}")
print("💡 In GitHub Codespaces: Check the 'PORTS' tab below for forwarded port 4040 to access Spark UI")

2026-02-09 21:00:35 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


🚀 Spark 3.5.0 - Query Optimization Lab
Spark UI: http://127.0.0.1:4040
💡 In GitHub Codespaces: Check the 'PORTS' tab below for forwarded port 4040 to access Spark UI


## Setup: Load IoT Sensor Data

We'll use IoT sensor readings to demonstrate query optimization techniques. This dataset contains sensor measurements from multiple buildings with various sensor types (temperature, humidity, motion, pressure).

In [2]:
# Load CSV data for initial examples
df = spark.read.csv("../Datasets/iot_sensor_readings.csv", header=True, inferSchema=True)

print("📊 IoT Sensor Dataset:")
print(f"Total records: {df.count():,}")
print(f"\nSchema:")
df.printSchema()
print(f"\nSample data:")
df.show(5, truncate=False)

📊 IoT Sensor Dataset:
Total records: 100,000

Schema:
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- location: string (nullable = true)
 |-- status: string (nullable = true)
 |-- battery_level: double (nullable = true)
 |-- signal_strength: double (nullable = true)


Sample data:
+------------+-------------------+------+-------+------------------+------+-----------------+-------------------+
|sensor_id   |timestamp          |value |unit   |location          |status|battery_level    |signal_strength    |
+------------+-------------------+------+-------+------------------+------+-----------------+-------------------+
|HUMID_002   |2025-08-29 11:59:00|48.62 |percent|Building_B_Floor_3|NORMAL|67.94829784734588|-52.00377612315141 |
|TEMP_002    |2025-08-29 12:02:00|7.37  |celsius|Building_B_Floor_1|NORMAL|77.10778082551532|-34.949122463877735|
|MOTION_001  |2025-08-29 12:

## Part 1: Logical Optimizations

Spark's **Catalyst optimizer** automatically optimizes query plans. Let's explore how it handles multiple filter transformations.

### Example 1.1: Multiple Chained Filters

It's natural to write filters separately for readability. Let's see what Catalyst does with multiple chained filters.

In [3]:
# Multiple chained filters - might seem inefficient
limit_sensors_df = (df
    .filter(col("sensor_id") != "TEMP_001")
    .filter(col("sensor_id") != "TEMP_002") 
    .filter(col("sensor_id") != "HUMID_001")
    .filter(col("sensor_id") != "HUMID_002")
    .filter(col("sensor_id") != "MOTION_001")
    .filter(col("location") != "Building_A_Floor_1")
    .filter(col("location") != "Building_A_Floor_2")
    .filter(col("location") != "Building_B_Floor_1")
)

print("Multiple chained filters - Explain plan:")
print("="*80)
limit_sensors_df.explain(True)

Multiple chained filters - Explain plan:
== Parsed Logical Plan ==
'Filter NOT ('location = Building_B_Floor_1)
+- Filter NOT (location#21 = Building_A_Floor_2)
   +- Filter NOT (location#21 = Building_A_Floor_1)
      +- Filter NOT (sensor_id#17 = MOTION_001)
         +- Filter NOT (sensor_id#17 = HUMID_002)
            +- Filter NOT (sensor_id#17 = HUMID_001)
               +- Filter NOT (sensor_id#17 = TEMP_002)
                  +- Filter NOT (sensor_id#17 = TEMP_001)
                     +- Relation [sensor_id#17,timestamp#18,value#19,unit#20,location#21,status#22,battery_level#23,signal_strength#24] csv

== Analyzed Logical Plan ==
sensor_id: string, timestamp: timestamp, value: double, unit: string, location: string, status: string, battery_level: double, signal_strength: double
Filter NOT (location#21 = Building_B_Floor_1)
+- Filter NOT (location#21 = Building_A_Floor_2)
   +- Filter NOT (location#21 = Building_A_Floor_1)
      +- Filter NOT (sensor_id#17 = MOTION_001)
        

**Key Observation**: Notice in the **Optimized Logical Plan** that Catalyst automatically consolidates all the filters into a single Filter operation with combined conditions!

### Example 1.2: Single Consolidated Filter

We could write it ourselves with a single filter. Let's compare the plans.

In [4]:
# Single consolidated filter - manually optimized
better_df = (df
    .filter(
        (col("sensor_id").isNotNull()) &
        (col("sensor_id") != "TEMP_001") &
        (col("sensor_id") != "TEMP_002") &
        (col("sensor_id") != "HUMID_001") &
        (col("sensor_id") != "HUMID_002") &
        (col("sensor_id") != "MOTION_001") &
        (col("location") != "Building_A_Floor_1") &
        (col("location") != "Building_A_Floor_2") &
        (col("location") != "Building_B_Floor_1")
    )
)

print("Single consolidated filter - Explain plan:")
print("="*80)
better_df.explain(True)

Single consolidated filter - Explain plan:
== Parsed Logical Plan ==
'Filter ((((((((isnotnull('sensor_id) AND NOT ('sensor_id = TEMP_001)) AND NOT ('sensor_id = TEMP_002)) AND NOT ('sensor_id = HUMID_001)) AND NOT ('sensor_id = HUMID_002)) AND NOT ('sensor_id = MOTION_001)) AND NOT ('location = Building_A_Floor_1)) AND NOT ('location = Building_A_Floor_2)) AND NOT ('location = Building_B_Floor_1))
+- Relation [sensor_id#17,timestamp#18,value#19,unit#20,location#21,status#22,battery_level#23,signal_strength#24] csv

== Analyzed Logical Plan ==
sensor_id: string, timestamp: timestamp, value: double, unit: string, location: string, status: string, battery_level: double, signal_strength: double
Filter ((((((((isnotnull(sensor_id#17) AND NOT (sensor_id#17 = TEMP_001)) AND NOT (sensor_id#17 = TEMP_002)) AND NOT (sensor_id#17 = HUMID_001)) AND NOT (sensor_id#17 = HUMID_002)) AND NOT (sensor_id#17 = MOTION_001)) AND NOT (location#21 = Building_A_Floor_1)) AND NOT (location#21 = Building_A_Flo

**Key Observation**: The **Optimized Logical Plan** is nearly identical! Catalyst produces the same optimized query regardless of how you write it. Write for readability - Catalyst handles optimization.

### Example 1.3: Duplicate Filters

In complex queries, you might accidentally duplicate filter conditions. Let's see how Catalyst handles this.

In [5]:
# Duplicate filters - accidentally filtering same condition multiple times
duplicate_df = (df
    .filter(col("status") != "ERROR")
    .filter(col("status") != "ERROR")  # Duplicate
    .filter(col("status") != "ERROR")  # Duplicate
    .filter(col("status") != "ERROR")  # Duplicate
    .filter(col("status") != "ERROR")  # Duplicate
)

print("Duplicate filters - Explain plan:")
print("="*80)
duplicate_df.explain(True)

print("\n" + "="*80)
print("Result: Catalyst eliminates duplicates and creates a single filter!")

Duplicate filters - Explain plan:
== Parsed Logical Plan ==
'Filter NOT ('status = ERROR)
+- Filter NOT (status#22 = ERROR)
   +- Filter NOT (status#22 = ERROR)
      +- Filter NOT (status#22 = ERROR)
         +- Filter NOT (status#22 = ERROR)
            +- Relation [sensor_id#17,timestamp#18,value#19,unit#20,location#21,status#22,battery_level#23,signal_strength#24] csv

== Analyzed Logical Plan ==
sensor_id: string, timestamp: timestamp, value: double, unit: string, location: string, status: string, battery_level: double, signal_strength: double
Filter NOT (status#22 = ERROR)
+- Filter NOT (status#22 = ERROR)
   +- Filter NOT (status#22 = ERROR)
      +- Filter NOT (status#22 = ERROR)
         +- Filter NOT (status#22 = ERROR)
            +- Relation [sensor_id#17,timestamp#18,value#19,unit#20,location#21,status#22,battery_level#23,signal_strength#24] csv

== Optimized Logical Plan ==
Filter (isnotnull(status#22) AND NOT (status#22 = ERROR))
+- Relation [sensor_id#17,timestamp#18,va

### Exercise 1.1: Filter Optimization Analysis

**Task**: Create two DataFrames:
1. One with 5 chained filters on different columns
2. One with the same conditions combined in a single filter

Use `explain()` to verify the optimized plans are identical, then confirm both produce the same count.

In [6]:
# Solution: Filter Optimization Exercise
print("🎯 Exercise 1.1: Filter Optimization Analysis\n")

# Approach 1: Chained filters
exercise_chained = (df
    .filter(col("status") == "NORMAL")
    .filter(col("unit") == "celsius")
    .filter(col("value") > 5)
    .filter(col("value") < 15)
    .filter(col("battery_level") > 70)
)

# Approach 2: Consolidated filter
exercise_consolidated = df.filter(
    (col("status") == "NORMAL") &
    (col("unit") == "celsius") &
    (col("value") > 5) &
    (col("value") < 15) &
    (col("battery_level") > 70)
)

# Show explain plans
print("Chained filters explain plan:")
exercise_chained.explain()

print("\n" + "="*80)
print("Consolidated filter explain plan:")
exercise_consolidated.explain()

# Verify same results
chained_count = exercise_chained.count()
consolidated_count = exercise_consolidated.count()

print("\n" + "="*80)
print(f"Chained approach: {chained_count:,} records")
print(f"Consolidated approach: {consolidated_count:,} records")

assert chained_count == consolidated_count, "Counts should match!"
print("\n✓ Exercise 1.1 completed! Both approaches produce identical optimized plans and results.")

🎯 Exercise 1.1: Filter Optimization Analysis

Chained filters explain plan:
== Physical Plan ==
*(1) Filter ((((((((isnotnull(status#22) AND isnotnull(unit#20)) AND isnotnull(value#19)) AND isnotnull(battery_level#23)) AND (status#22 = NORMAL)) AND (unit#20 = celsius)) AND (value#19 > 5.0)) AND (value#19 < 15.0)) AND (battery_level#23 > 70.0))
+- FileScan csv [sensor_id#17,timestamp#18,value#19,unit#20,location#21,status#22,battery_level#23,signal_strength#24] Batched: false, DataFilters: [isnotnull(status#22), isnotnull(unit#20), isnotnull(value#19), isnotnull(battery_level#23), (sta..., Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/workspaces/dscc202-402-spring2026/examples/Datasets/iot_sensor_r..., PartitionFilters: [], PushedFilters: [IsNotNull(status), IsNotNull(unit), IsNotNull(value), IsNotNull(battery_level), EqualTo(status,N..., ReadSchema: struct<sensor_id:string,timestamp:timestamp,value:double,unit:string,location:string,status:strin...



Consolidated filter expl

## Part 2: Understanding Caching

By default, DataFrame data exists on a Spark cluster only while being processed during a query. You can explicitly persist a DataFrame using the **`cache()`** method.

### When to Cache (Best Practices)

✅ **DO cache when**:
- **Exploratory data analysis**: Running multiple different queries on the same dataset
- **Machine learning**: Iteratively training models on the same data
- **Iterative algorithms**: Reusing the same DataFrame multiple times

❌ **DON'T cache when**:
- Using data only once (no benefit, wastes resources)
- Caching consumes cluster memory that could be used for task execution
- **Caching can prevent query optimizations** (like predicate pushdown, as we'll see next)

⚠️ **Important**: Always call **`unpersist()`** when done with cached data to free up memory!

In [7]:
# Example: When caching makes sense
print("Example: Multiple operations on cached data\n")

# Cache for iterative analysis
analysis_df = df.filter(col("status") == "NORMAL").cache()

# First action triggers caching
print(f"Total NORMAL readings: {analysis_df.count():,}")

# Subsequent operations use cached data (faster)
print(f"Average value: {analysis_df.agg({'value': 'avg'}).collect()[0][0]:.2f}")
print(f"Distinct sensors: {analysis_df.select('sensor_id').distinct().count()}")
print(f"Distinct locations: {analysis_df.select('location').distinct().count()}")

# Always unpersist when done
analysis_df.unpersist()
print("\n✓ Cache cleared")

Example: Multiple operations on cached data

Total NORMAL readings: 90,201
Average value: 192.31
Distinct sensors: 6
Distinct locations: 12

✓ Cache cleared


### Exercise 2.1: Caching Strategy Analysis

**Task**: Compare performance with and without caching for multiple queries on the same filtered dataset.

In [8]:
# Solution: Caching Strategy Exercise
print("🎯 Exercise 2.1: Caching Strategy Analysis\n")

# Test WITHOUT caching
print("Scenario 1: Without caching (recompute each time)")
base_df = df.filter(col("battery_level") > 50)

start = time.time()
query1 = base_df.count()
query2 = base_df.agg({'value': 'avg'}).collect()[0][0]
query3 = base_df.select('sensor_id').distinct().count()
time_without_cache = time.time() - start

print(f"Time: {time_without_cache:.4f}s")
print(f"Results: {query1:,} records, avg value: {query2:.2f}, {query3} sensors\n")

# Test WITH caching
print("Scenario 2: With caching (compute once, reuse)")
cached_base_df = df.filter(col("battery_level") > 50).cache()

start = time.time()
query1_cached = cached_base_df.count()
query2_cached = cached_base_df.agg({'value': 'avg'}).collect()[0][0]
query3_cached = cached_base_df.select('sensor_id').distinct().count()
time_with_cache = time.time() - start

print(f"Time: {time_with_cache:.4f}s")
print(f"Results: {query1_cached:,} records, avg value: {query2_cached:.2f}, {query3_cached} sensors\n")

# Compare
print("="*80)
if time_with_cache < time_without_cache:
    speedup = time_without_cache / time_with_cache
    print(f"✓ Caching provided {speedup:.2f}x speedup for multiple queries!")
else:
    print("Note: Speedup varies based on data size and number of reuses")

# Clean up
cached_base_df.unpersist()

assert query1 == query1_cached, "Results should match"
print("\n✓ Exercise 2.1 completed! Caching benefits multiple operations on same data.")

🎯 Exercise 2.1: Caching Strategy Analysis

Scenario 1: Without caching (recompute each time)
Time: 1.0862s
Results: 98,988 records, avg value: 192.53, 6 sensors

Scenario 2: With caching (compute once, reuse)
Time: 0.9151s
Results: 98,988 records, avg value: 192.53, 6 sensors

✓ Caching provided 1.19x speedup for multiple queries!

✓ Exercise 2.1 completed! Caching benefits multiple operations on same data.


## Part 3: Predicate Pushdown with Parquet

**Predicate pushdown** is a critical optimization where Spark pushes filter operations down to the data source, reducing the amount of data that needs to be read into memory.

### How it Works:
- **Parquet format** stores column statistics (min/max values) for each row group
- Spark can skip reading entire row groups that don't match filter conditions
- This dramatically reduces I/O and improves query performance

### What to Look for in Explain Plans:
- **`PushedFilters:`** Shows filters pushed to the data source
- **`FileScan parquet`** Shows the Parquet scan operation
- **No separate `Filter` operation** means filtering happens during read

In [9]:
# Load Parquet data
parquet_df = spark.read.parquet("../Datasets/iot_sensor_readings.parquet")

#print("📊 Loaded Parquet dataset")
#print(f"Total records: {parquet_df.count():,}")

### Example 3.1: Parquet Predicate Pushdown

Let's apply a filter and examine the explain plan to see predicate pushdown in action.

In [10]:
# Filter Parquet data - observe predicate pushdown
filtered_parquet = parquet_df.filter(col("status") == "NORMAL")

print("Parquet with Filter - Explain Plan:")
print("="*80)
filtered_parquet.explain(True)

print("\n" + "="*80)
print("🔍 Look for 'PushedFilters: [IsNotNull(status), EqualTo(status,NORMAL)]' in the scan!")
print("This means the filter is applied DURING the read, not after!")

Parquet with Filter - Explain Plan:
== Parsed Logical Plan ==
'Filter ('status = NORMAL)
+- Relation [sensor_id#1751,timestamp#1752,value#1753,unit#1754,location#1755,status#1756,battery_level#1757,signal_strength#1758] parquet

== Analyzed Logical Plan ==
sensor_id: string, timestamp: string, value: double, unit: string, location: string, status: string, battery_level: double, signal_strength: double
Filter (status#1756 = NORMAL)
+- Relation [sensor_id#1751,timestamp#1752,value#1753,unit#1754,location#1755,status#1756,battery_level#1757,signal_strength#1758] parquet

== Optimized Logical Plan ==
Filter (isnotnull(status#1756) AND (status#1756 = NORMAL))
+- Relation [sensor_id#1751,timestamp#1752,value#1753,unit#1754,location#1755,status#1756,battery_level#1757,signal_strength#1758] parquet

== Physical Plan ==
*(1) Filter (isnotnull(status#1756) AND (status#1756 = NORMAL))
+- *(1) ColumnarToRow
   +- FileScan parquet [sensor_id#1751,timestamp#1752,value#1753,unit#1754,location#1755,st

### Example 3.2: CSV vs Parquet Comparison

Let's compare Parquet (with pushdown) vs CSV (without pushdown) to see the difference.

In [11]:
# CSV does NOT support predicate pushdown
csv_df = spark.read.csv("../Datasets/iot_sensor_readings.csv", header=True, inferSchema=True)
filtered_csv = csv_df.filter(col("status") == "NORMAL")

print("CSV with Filter - Explain Plan:")
print("="*80)
filtered_csv.explain(True)

print("\n" + "="*80)
print("CSV must read the entire file, then apply the filter in a separate operation.")

CSV with Filter - Explain Plan:
== Parsed Logical Plan ==
'Filter ('status = NORMAL)
+- Relation [sensor_id#1785,timestamp#1786,value#1787,unit#1788,location#1789,status#1790,battery_level#1791,signal_strength#1792] csv

== Analyzed Logical Plan ==
sensor_id: string, timestamp: timestamp, value: double, unit: string, location: string, status: string, battery_level: double, signal_strength: double
Filter (status#1790 = NORMAL)
+- Relation [sensor_id#1785,timestamp#1786,value#1787,unit#1788,location#1789,status#1790,battery_level#1791,signal_strength#1792] csv

== Optimized Logical Plan ==
Filter (isnotnull(status#1790) AND (status#1790 = NORMAL))
+- Relation [sensor_id#1785,timestamp#1786,value#1787,unit#1788,location#1789,status#1790,battery_level#1791,signal_strength#1792] csv

== Physical Plan ==
*(1) Filter (isnotnull(status#1790) AND (status#1790 = NORMAL))
+- FileScan csv [sensor_id#1785,timestamp#1786,value#1787,unit#1788,location#1789,status#1790,battery_level#1791,signal_streng

### Example 3.3: Performance Comparison

Let's measure the actual performance difference between CSV and Parquet.

In [14]:
# Performance test: CSV vs Parquet
print("⚡ Performance Comparison: CSV vs Parquet\n")

# Test CSV
csv_df_test = spark.read.csv("../Datasets/iot_sensor_readings.csv", header=True, inferSchema=True)
start = time.time()
csv_filtered = csv_df_test.filter(
    (col("status") == "NORMAL") & 
    (col("value") > 50) & 
    (col("battery_level") > 60)
)
csv_count = csv_filtered.count()
csv_time = time.time() - start

# Test Parquet
parquet_df_test = spark.read.parquet("../Datasets/iot_sensor_readings.parquet")
start = time.time()
parquet_filtered = parquet_df_test.filter(
    (col("status") == "NORMAL") & 
    (col("value") > 50) & 
    (col("battery_level") > 60)
)
parquet_count = parquet_filtered.count()
parquet_time = time.time() - start

# Results
print(f"CSV:     {csv_count:,} records in {csv_time:.4f}s")
print(f"Parquet: {parquet_count:,} records in {parquet_time:.4f}s")
print(f"\nSpeedup: {csv_time/parquet_time:.2f}x faster with Parquet")
print(f"Savings: {((csv_time - parquet_time)/csv_time * 100):.1f}% time reduction")

assert csv_count == parquet_count, "Both should return same results"
print("\n✓ Results verified - same data, better performance!")

⚡ Performance Comparison: CSV vs Parquet

CSV:     28,717 records in 0.1632s
Parquet: 28,717 records in 0.0945s

Speedup: 1.73x faster with Parquet
Savings: 42.1% time reduction

✓ Results verified - same data, better performance!


### Exercise 3.1: Predicate Pushdown Analysis (ADDITIONAL)

**Task**: Create a complex filter with multiple conditions and analyze the predicate pushdown behavior for both CSV and Parquet formats.

In [ ]:
# Solution: Predicate Pushdown Exercise
print("🎯 Exercise 3.1: Predicate Pushdown Analysis\n")

# Complex filter conditions
complex_filter = (
    (col("sensor_id").startswith("TEMP")) &
    (col("value") >= 5) &
    (col("value") <= 10) &
    (col("battery_level") > 75) &
    (col("location").contains("Building_B"))
)

# Test with Parquet
print("Parquet with complex filter:")
parquet_complex = parquet_df.filter(complex_filter)
parquet_complex.explain()

print("\n" + "="*80)

# Test with CSV
print("CSV with same complex filter:")
csv_complex = csv_df.filter(complex_filter)
csv_complex.explain()

print("\n" + "="*80)

# Count results
parquet_result = parquet_complex.count()
csv_result = csv_complex.count()

print(f"\nParquet result: {parquet_result:,} records")
print(f"CSV result: {csv_result:,} records")

assert parquet_result == csv_result, "Both should return same results"

print("\n✓ Exercise 3.1 completed!")
print("Key insight: Parquet pushed numeric and equality filters to scan, CSV read everything first.")

🎯 Exercise 3.1: Predicate Pushdown Analysis

Parquet with complex filter:
== Physical Plan ==
*(1) Filter ((((((((isnotnull(sensor_id#1911) AND isnotnull(value#1913)) AND isnotnull(battery_level#1917)) AND isnotnull(location#1915)) AND StartsWith(sensor_id#1911, TEMP)) AND (value#1913 >= 5.0)) AND (value#1913 <= 10.0)) AND (battery_level#1917 > 75.0)) AND Contains(location#1915, Building_B))
+- *(1) ColumnarToRow
   +- FileScan parquet [sensor_id#1911,timestamp#1912,value#1913,unit#1914,location#1915,status#1916,battery_level#1917,signal_strength#1918] Batched: true, DataFilters: [isnotnull(sensor_id#1911), isnotnull(value#1913), isnotnull(battery_level#1917), isnotnull(locat..., Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspaces/dscc202-402-spring2026/examples/Datasets/iot_sensor_r..., PartitionFilters: [], PushedFilters: [IsNotNull(sensor_id), IsNotNull(value), IsNotNull(battery_level), IsNotNull(location), StringSta..., ReadSchema: struct<sensor_id:string,timesta

In [ ]:
#csv_complex.explain("formatted")
csv_result = csv_df.select("sensor_id", "value").count()
print(f"CSV result: {csv_result:,} records")

CSV result: 100,000 records


In [ ]:
#parquet_complex.explain()
parquet_result = parquet_df.select("sensor_id", "value").count()
print(f"\nParquet result: {parquet_result:,} records")


Parquet result: 100,000 records


## Part 4: Performance Impact of Caching

# Example 4.1: Performance Impact

Let's measure the performance difference between caching before vs after filtering.

In [17]:
# Performance comparison: Cache before vs after filtering
print("⚡ Performance Impact: Cache Before vs After Filtering\n")

# Clear any existing cache
spark.catalog.clearCache()

# Scenario 1: Cache before filtering (anti-pattern)
start = time.time()
df_cache_first = spark.read.parquet("../Datasets/iot_sensor_readings.parquet").cache()
df_cache_first.count()  # Materialize cache
result1 = df_cache_first.filter(
    (col("status") == "NORMAL") & (col("value") > 50)
).count()
time_cache_before = time.time() - start

print(f"Cache before filtering: {result1:,} results in {time_cache_before:.4f}s")

# Clear cache
df_cache_first.unpersist()
spark.catalog.clearCache()

# Scenario 2: Filter before caching (correct pattern)
start = time.time()
df_filter_first = (
    spark.read.parquet("../Datasets/iot_sensor_readings.parquet")
    .filter((col("status") == "NORMAL") & (col("value") > 50))
    .cache()
)
result2 = df_filter_first.count()  # Materialize cache
time_filter_before = time.time() - start

print(f"Filter before caching: {result2:,} results in {time_filter_before:.4f}s")

# Cleanup
df_filter_first.unpersist()
cached_parquet.unpersist()
parquet_filtered_first.unpersist()
spark.catalog.clearCache()

# Analysis
print("\n" + "="*80)
if time_filter_before < time_cache_before:
    improvement = ((time_cache_before - time_filter_before) / time_cache_before * 100)
    print(f"✓ Filtering first is {improvement:.1f}% faster!")
    print(f"Speedup: {time_cache_before/time_filter_before:.2f}x")
else:
    print("Note: Performance varies by data size and filter selectivity")

assert result1 == result2, "Both approaches should return same results"
print("\n✓ Performance comparison completed!")

⚡ Performance Impact: Cache Before vs After Filtering

Cache before filtering: 30,172 results in 0.4968s
Filter before caching: 30,172 results in 0.2477s

✓ Filtering first is 50.2% faster!
Speedup: 2.01x

✓ Performance comparison completed!


## Summary: Query Optimization Best Practices

### Key Concepts Mastered:

1. **Catalyst Optimizer**
   - Automatically consolidates multiple filters
   - Eliminates duplicate conditions
   - Write for readability - Catalyst handles efficiency

2. **Caching Strategy**
   - ✅ Cache for iterative operations (EDA, ML training)
   - ❌ Don't cache for single-use queries
   - ✅ Always `unpersist()` when done
   - ✅ Filter BEFORE caching to reduce memory usage

3. **Predicate Pushdown**
   - Parquet enables pushdown via column statistics
   - Look for `PushedFilters` in explain plans
   - 2-5x performance improvement typical
   - CSV requires full file scan, then filtering

4. **Keep in mind**
   - Caching loses predicate pushdown)
   - Instead of using CSV for large analytical queries, use Parquet
   - Caching rarely-used data wastes resources

### Performance Impact Summary:

| **Optimization** | **Impact** | **When to Use** |
|-----------------|-----------|----------------|
| Parquet vs CSV | 2-5x faster | Always for analytics |
| Predicate Pushdown | 3-10x faster | Filter on indexed columns |
| Caching (correct) | 2-3x faster | Multiple operations on same data |
| Filter before Cache | 50-80% memory savings | When caching is necessary |

### Best Practices:
1. ✅ Use Parquet for analytical workloads
2. ✅ Apply filters as early as possible
3. ✅ Let Catalyst optimize - write readable code
4. ✅ Cache only when reusing data multiple times
5. ✅ Filter before caching
6. ✅ Always unpersist() cached DataFrames
7. ✅ Use `explain()` to verify optimizations
8. ✅ Monitor Spark UI for query performance

---

**Next Steps**: Apply these optimization techniques to your data pipelines. Use `explain()` regularly to verify that Spark is optimizing your queries as expected!

In [ ]:
# Final cleanup
spark.catalog.clearCache()
spark.stop()

print("🎉 Lab: Query Optimization completed!")
print("\n✓ Learned: Catalyst optimization, caching strategies, predicate pushdown, and anti-patterns")
print("\n➡️  Next: Apply these techniques to optimize your Spark applications!")